In [2]:
import pandas as pd
df = pd.read_csv("../data/ai_resume_screening.csv")
df.head()

,years_experience,skills_match_score,education_level,project_count,resume_length,github_activity,shortlisted
0,6,84.7,Bachelors,7,234,158,No
1,3,59.1,Masters,5,502,77,No
2,12,100.0,Masters,12,753,381,Yes
3,14,66.8,High School,8,529,407,Yes
4,10,99.6,Bachelors,10,754,331,Yes


In [ ]:
df.shape
#The dataset has 30000 rows (resumes), 7 columns (6 features and 1 label)

(30000, 7)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   years_experience    30000 non-null  int64  
 1   skills_match_score  30000 non-null  float64
 2   education_level     30000 non-null  str    
 3   project_count       30000 non-null  int64  
 4   resume_length       30000 non-null  int64  
 5   github_activity     30000 non-null  int64  
 6   shortlisted         30000 non-null  str    
dtypes: float64(1), int64(4), str(2)
memory usage: 1.6 MB


In [4]:
df.describe()

,years_experience,skills_match_score,project_count,resume_length,github_activity
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000
mean,7.506567,73.682653,10.646267,572.584700,325.260667
std,4.624104,16.765909,4.634047,178.709918,159.951803
min,0.000000,0.500000,0.000000,150.000000,0.000000
25%,3.750000,62.100000,7.000000,441.000000,202.000000
50%,7.000000,74.300000,10.000000,574.000000,321.000000
75%,12.000000,86.500000,14.000000,709.000000,443.000000
max,15.000000,100.000000,25.000000,900.000000,842.000000


In [5]:
df.columns

Index(['years_experience', 'skills_match_score', 'education_level',
       'project_count', 'resume_length', 'github_activity', 'shortlisted'],
      dtype='str')

In [6]:
df["shortlisted"].value_counts()
#As we can see, the data is imbalanced (70% yes, 30% no)

shortlisted
Yes    20966
No      9034
Name: count, dtype: int64

In [7]:
df.isnull().sum()
#No missing values

years_experience      0
skills_match_score    0
education_level       0
project_count         0
resume_length         0
github_activity       0
shortlisted           0
dtype: int64

In [8]:
df.duplicated().sum()
#No duplicated rows

np.int64(0)

In [9]:
# Next step: encoding categorical variables (degree)

# Using one-hot encoding,  the education_level is divided into 3 columns representing 3 categories and the 4th category
# becomes the baseline so if all false meaning Bachelor is true

df = pd.get_dummies(df, columns=["education_level"], drop_first=True)
df.head()

,years_experience,skills_match_score,project_count,resume_length,github_activity,shortlisted,education_level_High School,education_level_Masters,education_level_PhD
0,6,84.7,7,234,158,No,False,False,False
1,3,59.1,5,502,77,No,False,True,False
2,12,100.0,12,753,381,Yes,False,True,False
3,14,66.8,8,529,407,Yes,True,False,False
4,10,99.6,10,754,331,Yes,False,False,False


In [10]:
# Now we convert the labels (shortlisted Yes/No) to binary

df["shortlisted"] = df["shortlisted"].map({
    "Yes": 1,
    "No": 0
})

df["shortlisted"].value_counts()

shortlisted
1    20966
0     9034
Name: count, dtype: int64

In [11]:
from sklearn.model_selection import train_test_split
# now we seperate the features and label
X = df.drop("shortlisted", axis=1)
y = df["shortlisted"]

#Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [12]:
#Feature Scaling , only done on training set because test set must remain unseen
import numpy as np

mean = np.mean(X_train, axis=0)
std = np.std(X_train, axis=0)

In [13]:
# we used the Z-score scaling
X_train_scaled = (X_train - mean) / std
X_test_scaled = (X_test - mean) / std

In [14]:
X_train_scaled.head()

,years_experience,skills_match_score,project_count,resume_length,github_activity,education_level_High School,education_level_Masters,education_level_PhD
29983,0.752942,1.054430,2.231293,0.801634,1.189259,-0.332716,1.358538,-0.33549
19618,0.536941,0.226326,0.074021,0.589320,0.820375,-0.332716,1.358538,-0.33549
12656,0.752942,-0.089427,1.152657,0.097645,0.676573,-0.332716,-0.736086,-0.33549
17431,-0.327061,-0.417094,-1.004614,-1.489126,-0.630150,-0.332716,1.358538,-0.33549
13839,1.184942,0.571866,-0.141706,0.427291,0.345203,3.005569,-0.736086,-0.33549


In [15]:
#Now we will use multiple ML models so we can compare them and choose
# the best one for our project.
# Decision Tree Model:

from sklearn.tree import DecisionTreeClassifier

dtree = DecisionTreeClassifier(random_state=55)

dtree.fit(X_train, y_train)

dt_predictions = dtree.predict(X_test)

In [16]:
#now evaluate

from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, dt_predictions))
print(classification_report(y_test, dt_predictions))

[[1410  397]
 [ 424 3769]]
              precision    recall  f1-score   support

           0       0.77      0.78      0.77      1807
           1       0.90      0.90      0.90      4193

    accuracy                           0.86      6000
   macro avg       0.84      0.84      0.84      6000
weighted avg       0.86      0.86      0.86      6000



In [17]:
# Random Forest Model:
from sklearn.ensemble import RandomForestClassifier

rfc = RandomForestClassifier(n_estimators=200)

rfc.fit(X_train, y_train)

rfc_pred = rfc.predict(X_test)

In [18]:
#Evaluation:
print(confusion_matrix(y_test, rfc_pred))
print(classification_report(y_test, rfc_pred))

[[1467  340]
 [ 263 3930]]
              precision    recall  f1-score   support

           0       0.85      0.81      0.83      1807
           1       0.92      0.94      0.93      4193

    accuracy                           0.90      6000
   macro avg       0.88      0.87      0.88      6000
weighted avg       0.90      0.90      0.90      6000



In [19]:
#Logistic Regression Model:
# NOTE: here we used X_train_scaled because in logistic regression
# if features have very different scales large numbers dominate the model
# because it is based on gradient descent.
from sklearn.linear_model import LogisticRegression

logmodel = LogisticRegression(max_iter=200)

logmodel.fit(X_train_scaled, y_train)

log_predictions = logmodel.predict(X_test_scaled)

In [20]:
#Evaluation:
print(confusion_matrix(y_test, log_predictions))
print(classification_report(y_test, log_predictions))

[[1505  302]
 [ 265 3928]]
              precision    recall  f1-score   support

           0       0.85      0.83      0.84      1807
           1       0.93      0.94      0.93      4193

    accuracy                           0.91      6000
   macro avg       0.89      0.88      0.89      6000
weighted avg       0.91      0.91      0.91      6000



Note: since our project is Binary Classification we tested our data on 3 different models we learned in class except linear regression since it works with regression and not classification
